# Exploratory Data Analysis
### Binary Fault Detection Dataset - IEEE SB GEHU

**Objective:** Comprehensive analysis of sensor data for fault detection

**Analysis Pipeline:**
- Dataset loading and initial inspection
- Data quality assessment
- Statistical profiling
- Target variable analysis
- Feature distribution visualization
- Correlation analysis
- Outlier detection

In [ ]:
# Import required packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configure visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

print('All libraries imported successfully')

## Dataset Loading and Initial Exploration

In [ ]:
# Load training dataset
training_data = pd.read_csv('TRAIN.csv')

# Display basic information
num_samples, num_features = training_data.shape
print(f'Dataset Dimensions: {num_samples} samples × {num_features} attributes')
print(f'\nFeature Range: F01 through F47 ({num_features - 1} sensor readings)')
print(f'Target Variable: Class (Binary: 0=Normal, 1=Fault)')
print(f'\nMemory Usage: {training_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

# Preview dataset
print('\nSample Records:')
training_data.head()

## Data Quality Assessment

In [ ]:
# Check data types
print('Data Type Distribution:')
dtype_counts = training_data.dtypes.value_counts()
for dtype, count in dtype_counts.items():
    print(f'  {dtype}: {count} columns')

# Identify missing values
null_counts = training_data.isnull().sum()
total_nulls = null_counts.sum()
print(f'\nMissing Values: {total_nulls}')
if total_nulls == 0:
    print('  Dataset is complete - no missing values detected')
else:
    print('  Columns with missing data:')
    print(null_counts[null_counts > 0].sort_values(ascending=False))

# Detect duplicate records
duplicate_count = training_data.duplicated().sum()
print(f'\nDuplicate Records: {duplicate_count}')
if duplicate_count > 0:
    duplicate_pct = (duplicate_count / num_samples) * 100
    print(f'  {duplicate_pct:.2f}% of dataset contains duplicates')
    print(f'  Recommendation: Remove duplicates during preprocessing')
else:
    print('  No duplicate records found')

## Statistical Summary and Profiling

In [ ]:
# Generate comprehensive statistics
stats_summary = training_data.describe().T

# Add additional metrics
stats_summary['variance'] = training_data.var()
stats_summary['range'] = stats_summary['max'] - stats_summary['min']
stats_summary['iqr'] = stats_summary['75%'] - stats_summary['25%']
stats_summary['cv'] = (stats_summary['std'] / stats_summary['mean']).abs()

# Display formatted statistics
print('Comprehensive Statistical Profile:\n')
display_cols = ['mean', 'std', 'min', '25%', '50%', '75%', 'max', 'range', 'iqr']
stats_summary[display_cols].round(4).head(15)

## Target Variable Analysis

In [ ]:
# Analyze class distribution
target_distribution = training_data['Class'].value_counts().sort_index()
target_percentages = training_data['Class'].value_counts(normalize=True).sort_index() * 100

# Calculate imbalance ratio
majority_class = target_distribution.max()
minority_class = target_distribution.min()
imbalance_ratio = majority_class / minority_class

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
color_palette = ['#3498db', '#e74c3c']
bars = ax1.bar(target_distribution.index.astype(str), target_distribution.values, 
               color=color_palette, alpha=0.8, edgecolor='black', linewidth=1.5)

# Add value labels
for idx, (bar, count, pct) in enumerate(zip(bars, target_distribution.values, target_percentages.values)):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 100,
             f'{count:,}\n({pct:.1f}%)',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

ax1.set_xlabel('Class Label', fontsize=12, fontweight='bold')
ax1.set_ylabel('Sample Count', fontsize=12, fontweight='bold')
ax1.set_title('Class Distribution - Absolute Counts', fontsize=13, fontweight='bold')
ax1.set_xticklabels(['Class 0\n(Normal)', 'Class 1\n(Fault)'])
ax1.grid(axis='y', alpha=0.3)

# Pie chart
wedges, texts, autotexts = ax2.pie(target_distribution.values, 
                                     labels=['Normal\nOperation', 'Fault\nDetected'],
                                     autopct='%1.1f%%',
                                     colors=color_palette,
                                     startangle=90,
                                     explode=(0.05, 0.05),
                                     shadow=True,
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Class Distribution - Proportions', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable Analysis', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print analysis
print(f'\nClass Balance Analysis:')
print(f'  Imbalance Ratio: {imbalance_ratio:.2f}:1')
if imbalance_ratio > 2.5:
    print(f'  Significant class imbalance detected')
    print(f'  Consider: SMOTE, class weights, or stratified sampling')
elif imbalance_ratio > 1.5:
    print(f'  Moderate imbalance - monitor model performance')
else:
    print(f'  Classes are well balanced')

## Feature Distribution Analysis

In [ ]:
# Select feature columns
feature_cols = [col for col in training_data.columns if col.startswith('F')]
num_plots = len(feature_cols)

# Create subplot grid
n_cols = 6
n_rows = (num_plots + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 3))
axes = axes.flatten()

# Plot histograms
for idx, feature in enumerate(feature_cols):
    ax = axes[idx]
    
    # Calculate skewness
    skew_value = training_data[feature].skew()
    
    # Plot histogram
    ax.hist(training_data[feature], bins=40, color='steelblue', 
            alpha=0.7, edgecolor='black', linewidth=0.5)
    
    # Add vertical line for mean
    mean_val = training_data[feature].mean()
    ax.axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}')
    
    # Styling
    ax.set_title(f'{feature}\nSkew: {skew_value:.2f}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Value', fontsize=8)
    ax.set_ylabel('Frequency', fontsize=8)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(alpha=0.3)

# Hide unused subplots
for idx in range(num_plots, len(axes)):
    axes[idx].axis('off')

plt.suptitle('Feature Distribution Analysis (All Sensors)', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f'\nVisualized {num_plots} feature distributions')

## Skewness Analysis

In [ ]:
# Calculate skewness for all features
skewness_values = training_data[feature_cols].skew().sort_values(ascending=False)

# Categorize by skewness level
highly_skewed = skewness_values[skewness_values.abs() > 1]
moderately_skewed = skewness_values[(skewness_values.abs() > 0.5) & (skewness_values.abs() <= 1)]
low_skewed = skewness_values[skewness_values.abs() <= 0.5]

print(f'Skewness Distribution:')
print(f'  Highly Skewed (|skew| > 1): {len(highly_skewed)} features')
print(f'  Moderately Skewed (0.5 < |skew| <= 1): {len(moderately_skewed)} features')
print(f'  Low Skewness (|skew| <= 0.5): {len(low_skewed)} features')

# Visualization
fig, ax = plt.subplots(figsize=(14, 8))

# Color code by skewness level
colors = ['#e74c3c' if abs(x) > 1 else '#f39c12' if abs(x) > 0.5 else '#27ae60' 
          for x in skewness_values]

bars = ax.barh(range(len(skewness_values)), skewness_values.values, color=colors, alpha=0.7)
ax.set_yticks(range(len(skewness_values)))
ax.set_yticklabels(skewness_values.index, fontsize=8)
ax.axvline(0, color='black', linewidth=1)
ax.axvline(-1, color='gray', linestyle='--', alpha=0.5, label='|skew| = 1')
ax.axvline(1, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Skewness Value', fontsize=12, fontweight='bold')
ax.set_title('Feature Skewness Analysis (Sorted)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nRecommendation: Apply power transformation (Yeo-Johnson) to highly skewed features')

## Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = training_data[feature_cols].corr()

# Find highly correlated pairs
high_corr_threshold = 0.9
high_corr_pairs = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > high_corr_threshold:
            high_corr_pairs.append((
                correlation_matrix.columns[i],
                correlation_matrix.columns[j],
                correlation_matrix.iloc[i, j]
            ))

print(f'Highly Correlated Feature Pairs (|r| > {high_corr_threshold}):')
print(f'  Found {len(high_corr_pairs)} pairs\n')

if high_corr_pairs:
    for feat1, feat2, corr_val in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True)[:10]:
        print(f'  {feat1} <-> {feat2}: r = {corr_val:.4f}')
    print(f'\nConsider removing redundant features during preprocessing')

# Heatmap visualization
fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot=False, fmt='.2f', ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

## Outlier Detection

In [ ]:
# Detect outliers using IQR method
outlier_summary = []

for feature in feature_cols:
    Q1 = training_data[feature].quantile(0.25)
    Q3 = training_data[feature].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = training_data[(training_data[feature] < lower_bound) | 
                             (training_data[feature] > upper_bound)]
    
    if len(outliers) > 0:
        outlier_summary.append({
            'Feature': feature,
            'Outlier_Count': len(outliers),
            'Outlier_Pct': (len(outliers) / num_samples) * 100,
            'Lower_Bound': lower_bound,
            'Upper_Bound': upper_bound
        })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier_Count', ascending=False)

print(f'Outlier Detection Summary (IQR Method):')
print(f'  Features with outliers: {len(outlier_df)}/{len(feature_cols)}')
print(f'  Total outlier instances: {outlier_df["Outlier_Count"].sum():,}\n')

print('  Top 10 features by outlier count:')
outlier_df.head(10)

## Summary and Recommendations

In [ ]:
print('='*70)
print('EXPLORATORY DATA ANALYSIS - KEY FINDINGS')
print('='*70)

print(f'\n1. DATASET OVERVIEW:')
print(f'   - Total Samples: {num_samples:,}')
print(f'   - Total Features: {len(feature_cols)}')
print(f'   - Missing Values: {total_nulls}')
print(f'   - Duplicate Records: {duplicate_count:,}')

print(f'\n2. TARGET VARIABLE:')
print(f'   - Class 0 (Normal): {target_distribution[0]:,} ({target_percentages[0]:.1f}%)')
print(f'   - Class 1 (Fault): {target_distribution[1]:,} ({target_percentages[1]:.1f}%)')
print(f'   - Imbalance Ratio: {imbalance_ratio:.2f}:1')

print(f'\n3. FEATURE CHARACTERISTICS:')
print(f'   - Highly Skewed Features: {len(highly_skewed)}')
print(f'   - Highly Correlated Pairs: {len(high_corr_pairs)}')
print(f'   - Features with Outliers: {len(outlier_df)}')

print(f'\n4. PREPROCESSING RECOMMENDATIONS:')
print(f'   - Remove {duplicate_count:,} duplicate records')
print(f'   - Apply outlier capping (Winsorization at 1st/99th percentile)')
print(f'   - Transform {len(highly_skewed)} highly skewed features (Yeo-Johnson)')
print(f'   - Consider removing {len(high_corr_pairs)} redundant features')
print(f'   - Apply StandardScaler normalization')
if imbalance_ratio > 2:
    print(f'   - Address class imbalance (SMOTE/class weights)')

print('\n' + '='*70)
print('Exploratory Data Analysis Complete')
print('='*70)